# InjectNo — Notebook 12: Safety-drift check with Lexsi **SafeTune**
Fine-tuning can quietly weaken a model's general safety ("safety drift"). We check whether InjectNo's safety changed:
the original Aya Expanse 8B and InjectNo (same base + our LoRA adapter) answer the same prompts as a plain chat model
(no agent prompt), and SafeTune's `evaluate()` scores them with the **WildGuard** judge:

- **HarmBench** (standard behaviours, first 100): harmful requests. Metric `asr` = fraction answered instead of refused (**lower = safer**).
- **OR-Bench hard** (first 100): safe prompts that *look* risky. Metric `refusal_rate` (**lower = less over-refusal**).

Memory: the 8B model (4-bit) runs on GPU 0 and the WildGuard judge (8-bit) on GPU 1, so **use GPU T4 x2**.
Before running: accept the terms of [walledai/HarmBench](https://huggingface.co/datasets/walledai/HarmBench) and
[allenai/wildguard](https://huggingface.co/allenai/wildguard) on Hugging Face with the same account.

**Settings:** GPU T4 x2, Internet On, Secret HF_TOKEN. Save Version -> Save & Run All (Commit). ~1-1.5 h.

In [1]:
!pip install -q safetune==0.1.12 bitsandbytes
!pip install -q -U "transformers>=5.15,<6" peft accelerate
import torch; print(torch.__version__, "GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, "This notebook needs GPU T4 x2 (judge on the second GPU)." 

2.11.0+cu130 GPUs: 2


In [2]:
import os
from huggingface_hub import login, whoami
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN")
login(HF_TOKEN)
HF_USER = whoami()["name"]; print("Logged in as", HF_USER)

Logged in as kenil1710


## 1. Access check (fails early with a clear message instead of after an hour)

In [3]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
for repo, kind in [("walledai/HarmBench", "dataset"), ("bench-llm/or-bench", "dataset"), ("allenai/wildguard", "model"),
                   ("CohereLabs/aya-expanse-8b", "model"), (f"{HF_USER}/injectno-aya-expanse-8b-v3", "model")]:
    try:
        api.list_repo_files(repo, repo_type=kind); print("OK  ", repo)
    except Exception as e:
        print("NO ACCESS", repo, "->", type(e).__name__, "| open https://huggingface.co/" + ("datasets/" if kind == "dataset" else "") + repo, "and accept the terms")

OK   walledai/HarmBench
OK   bench-llm/or-bench
OK   allenai/wildguard
OK   CohereLabs/aya-expanse-8b
OK   kenil1710/injectno-aya-expanse-8b-v3


## 2. Models: Aya Expanse 8B (4-bit, GPU 0) + InjectNo adapter

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
BASE = "CohereLabs/aya-expanse-8b"; ADAPTER = f"{HF_USER}/injectno-aya-expanse-8b-v3"
tok = AutoTokenizer.from_pretrained(BASE, token=HF_TOKEN)
if tok.pad_token is None: tok.pad_token = tok.eos_token
bnb4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb4, dtype=torch.float16, device_map={"": 0}, token=HF_TOKEN)
model = PeftModel.from_pretrained(base, ADAPTER, token=HF_TOKEN).eval()
print("loaded; GPU0 mem GB:", round(torch.cuda.memory_allocated(0) / 1e9, 1))

loaded; GPU0 mem GB: 5.9


## 3. SafeTune setup: WildGuard judge in 8-bit on GPU 1

In [5]:
import sys, importlib, safetune
from transformers import AutoTokenizer as _AT, AutoModelForCausalLM as _AM
ste = importlib.import_module("safetune.instrumentation.evaluate.suite.evaluate")

def load_judge_8bit(model_id):
    """Same as SafeTune's judge loader, but 8-bit on the second GPU (a 7B judge in fp16 does not fit next to the 8B model)."""
    jt = _AT.from_pretrained(model_id, token=HF_TOKEN)
    jt.padding_side = "left"
    if jt.pad_token is None: jt.pad_token = jt.eos_token
    jm = _AM.from_pretrained(model_id, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map={"": 1}, token=HF_TOKEN).eval()
    if jm.config.pad_token_id is None:
        jm.config.pad_token_id = jt.pad_token_id or jt.eos_token_id
        if hasattr(jm, "generation_config"): jm.generation_config.pad_token_id = jm.config.pad_token_id
    return jt, jm

ste._load_hf_model = load_judge_8bit
N_PROMPTS = 100
safetune.configure(max_prompts=N_PROMPTS, gen_batch_size=8, safety_max_new_tokens=256, eval_strict=False,
                   datasets={"harmbench": {"config": "standard"}})
BENCH = ["harmbench", "orbench_hard"]
print("benchmarks:", BENCH, "| prompts each:", N_PROMPTS)

I0000 00:00:1794049197.376477      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Overriding method-intrinsic setting(s) safety_max_new_tokens: results deviate from the methods' published defaults.


benchmarks: ['harmbench', 'orbench_hard'] | prompts each: 100


## 4. Run: original model (adapter off) vs InjectNo (adapter on)

In [6]:
import time, json
RES = {}
t0 = time.time()
with model.disable_adapter():
    RES["Aya Expanse 8B (original)"] = safetune.evaluate.evaluate(model, benchmarks=BENCH, judge="wildguard", tokenizer=tok,
                                                                  generation_kwargs={"do_sample": False})
print(f"original done in {(time.time()-t0)/60:.1f} min")
t0 = time.time()
RES["InjectNo (run 5)"] = safetune.evaluate.evaluate(model, benchmarks=BENCH, judge="wildguard", tokenizer=tok,
                                                     generation_kwargs={"do_sample": False})
print(f"InjectNo done in {(time.time()-t0)/60:.1f} min")

MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
[... previous line repeated 58239 more times]


MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
[... previous line repeated 58239 more times]


original done in 24.8 min


MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
[... previous line repeated 58239 more times]


MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
[... previous line repeated 58239 more times]


InjectNo done in 23.8 min


## 5. Results

In [7]:
import pandas as pd
rows = []
for m, r in RES.items():
    for b, v in r.items():
        if "error" in v:
            rows.append(dict(model=m, benchmark=b, error=v["error"])); continue
        rows.append(dict(model=m, benchmark=b, n=v.get("n_evaluated"), asr=round(v.get("asr", float("nan")), 3),
                         refusal_rate=round(v.get("refusal_rate", float("nan")), 3), headline=v.get("headline_metric")))
df = pd.DataFrame(rows); display(df)
try:
    hb = df[df.benchmark == "harmbench"].set_index("model").asr
    orr = df[df.benchmark == "orbench_hard"].set_index("model").refusal_rate
    print(f"HarmBench harmful-answer rate (lower = safer): original {hb.iloc[0]:.1%} -> InjectNo {hb.iloc[1]:.1%}")
    print(f"OR-Bench-hard over-refusal (lower = more helpful): original {orr.iloc[0]:.1%} -> InjectNo {orr.iloc[1]:.1%}")
except Exception as e:
    print("summary skipped:", e)

,model,benchmark,n,asr,refusal_rate,headline
0,Aya Expanse 8B (original),harmbench,100,0.23,0.77,asr
1,Aya Expanse 8B (original),orbench_hard,100,0.83,0.17,refusal_rate
2,InjectNo (run 5),harmbench,100,0.12,0.88,asr
3,InjectNo (run 5),orbench_hard,100,0.74,0.26,refusal_rate


HarmBench harmful-answer rate (lower = safer): original 23.0% -> InjectNo 12.0%
OR-Bench-hard over-refusal (lower = more helpful): original 17.0% -> InjectNo 26.0%


In [8]:
df.to_csv("injectno_safetune_drift.csv", index=False)
json.dump({m: {b: {k: v for k, v in r.items() if k in ("asr", "refusal_rate", "harmfulness_score", "n_evaluated", "n_total", "headline_metric", "error")}
              for b, r in res.items()} for m, res in RES.items()}, open("injectno_safetune_drift.json", "w"), indent=1)
for f in ("injectno_safetune_drift.csv", "injectno_safetune_drift.json"):
    try:
        api.upload_file(path_or_fileobj=f, path_in_repo=f"results/safetune/{f}", repo_id=f"{HF_USER}/injectno-bench", repo_type="dataset"); print("uploaded", f)
    except Exception as e: print("upload failed", f, repr(e)[:200])

uploaded injectno_safetune_drift.csv
uploaded injectno_safetune_drift.json
